In [6]:
import os
import numpy as np
import librosa

# =========================
# Configuration
# =========================
BASE_DIR = "/kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection"  # change to your dataset path
OUTPUT_DIR = "processed"

SR = 16000
DURATION = 10
TARGET_LEN = SR * DURATION

N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512

N_SELECTED_FRAMES = 250   # Pump rule (stationary → highest covariance)


# =========================
# Audio Loading
# =========================
def load_audio(file_path):
    y, _ = librosa.load(file_path, sr=SR)

    if len(y) > TARGET_LEN:
        y = y[:TARGET_LEN]
    else:
        y = np.pad(y, (0, TARGET_LEN - len(y)))

    return y


# =========================
# Log-Mel Extraction
# =========================
def extract_logmel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )

    logmel = librosa.power_to_db(mel)
    return logmel.T   # shape → (frames, 128)


# =========================
# Spectrogram Frame Selection
# =========================
def compute_frame_correlations(X):
    # X shape: (frames, 128)
    C = np.cov(X)              # (frames, frames)
    r = np.sum(C, axis=1)      # correlation magnitude
    return r


def select_top_frames(X, r, n=N_SELECTED_FRAMES):
    indices = np.argsort(r)[-n:]   # highest covariance
    return X[indices]


# =========================
# Complete File Processing
# =========================
def preprocess_file(file_path):
    y = load_audio(file_path)
    X = extract_logmel(y)

    r = compute_frame_correlations(X)
    selected = select_top_frames(X, r)

    return selected   # shape (250, 128)


# =========================
# Folder Processing
# =========================
def process_folder(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)

    for file in os.listdir(input_folder):
        if file.endswith(".wav"):
            input_path = os.path.join(input_folder, file)

            print(f"Processing: {input_path}")

            processed = preprocess_file(input_path)

            output_file = os.path.splitext(file)[0] + ".npy"
            output_path = os.path.join(output_folder, output_file)

            np.save(output_path, processed)


# =========================
# Main Execution
# =========================


subfolders = ["train-normal", "test-normal", "anomaly"]

for sub in subfolders:
    input_path = os.path.join(BASE_DIR, sub)
    output_path = os.path.join(OUTPUT_DIR, sub)

    if os.path.exists(input_path):
        process_folder(input_path, output_path)

print("Preprocessing complete.")

Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection/train-normal/normal_id_02_00000546.wav
Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection/train-normal/normal_id_02_00000764.wav
Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection/train-normal/normal_id_00_00000530.wav
Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection/train-normal/normal_id_02_00000820.wav
Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection/train-normal/normal_id_02_00000102.wav
Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water pump audio for anomaly detection/train-normal/normal_id_02_00000838.wav
Processing: /kaggle/input/anomaly-detection-in-water-pump-using-audio-data/water p

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# =========================
# Config
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 128
EPOCHS = 100
LR = 0.005          # slightly higher
BETA = 5.0          # emphasize reconstruction more
DATA_DIR = "processed"


# =========================
# Dataset
# =========================
class PumpDataset(Dataset):
    def __init__(self, folder):
        print(f"\nLoading data from: {folder}")

        self.files = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".npy")
        ]

        data_list = []

        for f in tqdm(self.files, desc="Reading files"):
            x = np.load(f)
            data_list.append(x)

        self.data = np.vstack(data_list)

        print("Normalizing features...")
        self.scaler = StandardScaler()
        self.data = self.scaler.fit_transform(self.data)

        self.data = torch.tensor(self.data, dtype=torch.float32)

        print(f"Total frames loaded: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# =========================
# AEFS Model
# =========================
class AEFS(nn.Module):
    def __init__(self):
        super(AEFS, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 8),
            nn.BatchNorm1d(8),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Linear(32, 128)
        )

        self.scaling_gate = nn.Parameter(torch.zeros(1, 128))

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        scaled = torch.exp(self.scaling_gate) * out
        return scaled


# =========================
# Training
# =========================
def train_model():
    train_dataset = PumpDataset(os.path.join(DATA_DIR, "train-normal"))
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True
    )

    model = AEFS().to(DEVICE)
    optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
    )
    best_loss = float("inf")
    patience = 10
    counter = 0
    mse = nn.MSELoss()

    print("\nStarting training...\n")

    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

        for batch in loop:
            batch = batch.to(DEVICE)

            recon = model(batch)

            L_rec = mse(recon, batch)
            L_sg = -torch.sum(model.scaling_gate)

            loss = BETA * L_rec + L_sg

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            counter = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            counter += 1
        
        if counter >= patience:
            print("Early stopping triggered")
            break

        if (epoch + 1) % 500 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS}, Total Loss: {epoch_loss:.4f}")

    return model, train_dataset.scaler



# =========================
# Main
# =========================

model, scaler = train_model()


Loading data from: processed/train-normal


Reading files: 100%|██████████| 2241/2241 [00:00<00:00, 4612.30it/s]


Normalizing features...
Total frames loaded: 560250

Starting training...



Epoch 2/100:  94%|█████████▍| 4109/4377 [00:27<00:01, 139.14it/s, loss=-1.22e+3]

In [ ]:

# =========================
# Evaluation
# =========================
def evaluate(model, scaler):
    model.eval()
    mse = nn.MSELoss(reduction='none')

    def compute_scores(folder, label):
        files = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.endswith(".npy")
        ]

        scores = []
        labels = []

        for f in tqdm(files, desc=f"Evaluating {folder}"):
            x = np.load(f)
            x = scaler.transform(x)
            x = torch.tensor(x, dtype=torch.float32).to(DEVICE)

            with torch.no_grad():
                recon = model(x)
                error = mse(recon, x).mean(dim=1)
                score = error.mean().item()

            scores.append(score)
            labels.append(label)

        return scores, labels


    print("\nEvaluating on test-normal...")
    normal_scores, normal_labels = compute_scores(
        os.path.join(DATA_DIR, "test-normal"), 0)

    print("\nEvaluating on anomaly...")
    anomaly_scores, anomaly_labels = compute_scores(
        os.path.join(DATA_DIR, "anomaly"), 1)

    scores = normal_scores + anomaly_scores
    labels = normal_labels + anomaly_labels

    threshold = np.mean(normal_scores) + 3 * np.std(normal_scores)
    preds = [1 if s > threshold else 0 for s in scores]

    print("\nClassification Report:\n")
    print(classification_report(labels, preds))

    auc = roc_auc_score(labels, scores)
    print("ROC-AUC Score:", auc)

evaluate(model, scaler)